# 08_discrete_time_hazard_model.ipynb — Discrete-Time Hazard Model

**แนวคิด**: แทนที่จะทำนาย readmission ของแต่ละ encounter แบบ i.i.d. เหมือน pipeline
เดิม (`04_model_training` → `05_improved_pipeline4`) โน้ตบุ๊กนี้เพิ่มมุมมองแบบ
**discrete-time hazard model** (person-period logistic regression ตามแนวทาง
Allison, 1982): มอง encounter แต่ละครั้งเป็น "period" หนึ่ง แล้วโมเดล hazard
(= P(readmit <30 วัน)) เป็นฟังก์ชันของ covariates เดิม **บวก** ตัวแปร
`visit_period` ที่บอกว่านี่คือ encounter ที่เท่าไหร่ของผู้ป่วยคนนี้ในชุดข้อมูล
(1 / 2 / 3 / 4+) — เป็นตัวแทนของ "duration dependence" ซึ่งเป็นหัวใจของโมเดล
แบบ hazard

⚠️ **ข้อควรระวังเรื่อง path**: โน้ตบุ๊กนี้รันจาก working directory ที่ notebook
ไฟล์วางอยู่ (เช่น `pj seminar/`) และคาดว่ามีโฟลเดอร์ `data/` อยู่ **ในระดับเดียวกัน**
(ไม่ใช่ `../data/`) — ถ้า path ไม่ตรงให้เช็คด้วย `os.getcwd()` และ `os.listdir(".")`
ก่อนแก้ path เอง

## 1. Import

In [1]:
import os
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, confusion_matrix


## 2. เช็ค Working Directory ก่อนเสมอ

รันเซลล์นี้ก่อนทุกครั้งที่เปิดโน้ตบุ๊กใหม่ เพื่อยืนยันว่า path ที่ใช้ทั้งไฟล์
ตรงกับตำแหน่งจริงของโฟลเดอร์ `data/`

In [2]:
print("Current working directory:", os.getcwd())
print("Files here:", os.listdir("."))
assert os.path.isdir("data"), (
    "ไม่พบโฟลเดอร์ data/ ใน working directory ปัจจุบัน — เช็ค path ใหม่ก่อนรันต่อ"
)
assert os.path.isfile("data/diabetic_data.csv"), (
    "ไม่พบไฟล์ data/diabetic_data.csv — วางไฟล์ raw dataset ไว้ในโฟลเดอร์ data/ ก่อน"
)
print("OK: พบ data/diabetic_data.csv พร้อมใช้งาน")


Current working directory: c:\Users\HP\seminar-final\notebook
Files here: ['01_data_understanding.ipynb', '02_preprocessing.ipynb', '03_model_preparation.ipynb', '04_model_training.ipynb', '05_improved_pipeline4.ipynb', '06_new_experiments.ipynb', '07_final_calibrated_pipeline.ipynb', '08_discrete_time_hazard_model2.ipynb', '08_feature_enrichment.ipynb', '08_paper_replication_lu_uddin.ipynb', '09_bootstrap_confidence_intervals.ipynb', 'catboost_info', 'confusion_matrix_final.png', 'fairness_ci_by_age.png', 'km_curve_by_risk_tier.png', 'patch_groupkfold_05_improved_pipeline4.md', 'result.ipynb']


AssertionError: ไม่พบโฟลเดอร์ data/ ใน working directory ปัจจุบัน — เช็ค path ใหม่ก่อนรันต่อ

## 3. Load & Clean Data

เหมือนกับ `02_preprocessing.ipynb` เป๊ะ (ตัด gender ผิดปกติ, ตัด
death/hospice discharge) **เพิ่มเติมจากเดิม**: sort ตาม `patient_nbr` +
`encounter_id` แล้วสร้าง `visit_number` = ลำดับ encounter ของผู้ป่วยคนนั้น
(1, 2, 3, ...)

⚠️ **กัน leakage**: `visit_number` นับเฉพาะ encounter ที่เกิดขึ้น "จนถึงและ
รวม" ครั้งนี้เท่านั้น ไม่แตะข้อมูล encounter ในอนาคตของผู้ป่วยคนเดียวกัน จึง
ใช้เป็น feature ได้อย่างปลอดภัย (ต่างจาก "จำนวน encounter ทั้งหมดของผู้ป่วย"
ซึ่งจะ leak อนาคต)

In [ ]:
df = pd.read_csv("data/diabetic_data.csv", keep_default_na=False, na_values=["?"])
print("Raw shape:", df.shape)

df = df[df["gender"] != "Unknown/Invalid"]

death_hospice_codes = [11, 13, 14, 19, 20, 21]
df = df[~df["discharge_disposition_id"].isin(death_hospice_codes)]
print("After removing death/hospice discharges:", df.shape)

# --- sort ตามลำดับเวลา (proxy) แล้วสร้าง visit_number ---
df = df.sort_values(["patient_nbr", "encounter_id"]).reset_index(drop=True)
df["visit_number"] = df.groupby("patient_nbr").cumcount() + 1
print("Unique patients:", df["patient_nbr"].nunique())
print(df["visit_number"].value_counts().sort_index().head(10))


C:\Users\ACER\AppData\Local\Temp\ipykernel_8012\3450235663.py:1: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("data/diabetic_data.csv", keep_default_na=False, na_values=["?"])


Raw shape: (101766, 50)
After removing death/hospice discharges: (99340, 50)
Unique patients: 69987
visit_number
1     69987
2     16341
3      6123
4      2890
5      1536
6       845
7       507
8       315
9       203
10      136
Name: count, dtype: int64


## 4. Feature Engineering (เหมือนเดิม + visit_period)

In [ ]:
df["total_prior_visits"] = (
    df["number_outpatient"] + df["number_emergency"] + df["number_inpatient"]
)
df["had_prior_inpatient"] = (df["number_inpatient"] > 0).astype(int)

# --- bucket visit_number เป็น period dummy (1 / 2 / 3 / 4+) ---
# เหตุผลที่ bucket แทนที่จะใช้ raw visit_number ตรงๆ: visit >= 4 มีจำนวนน้อยลง
# เรื่อยๆ (long tail) การรวมเป็น "4+" ช่วยให้แต่ละ period มีขนาดตัวอย่างพอ
# สำหรับประมาณค่า hazard ต่อ period ได้เสถียร
def bucket_visit(v):
    return "4+" if v >= 4 else str(v)

df["visit_period"] = df["visit_number"].apply(bucket_visit)
print(df["visit_period"].value_counts())


visit_period
1     69987
2     16341
4+     6889
3      6123
Name: count, dtype: int64


## 5. Drop / Reduce Columns, Diagnosis Grouping, Target, Save

เหมือนเดิมทุกจุดกับ `02_preprocessing.ipynb` **บวก**ขั้นตอนสุดท้ายคือเซฟผลลัพธ์
เป็น `data/diabetic_data_hazard.csv` เพื่อให้หัวข้อ 6 อ่านกลับเข้ามาใช้ต่อ

⚠️ **สำคัญ**: path ตอนเซฟต้องเป็น `"data/diabetic_data_hazard.csv"` (ไม่มี `../`)
ให้ตรงกับ path ตอนโหลด raw data ด้านบน — ถ้าใส่ `../` ไฟล์จะไปโผล่ผิดโฟลเดอร์
แล้วหัวข้อ 6 จะหาไฟล์ไม่เจอ

In [ ]:
# encounter_id ใช้ประโยชน์ในการ sort ไปแล้ว ตัดออกได้ตอนนี้เหมือนเดิม
drop_cols = ["weight", "payer_code", "encounter_id"]
df.drop(columns=drop_cols, inplace=True)

df["medical_specialty"] = df["medical_specialty"].fillna("Missing")
top_specialties = df["medical_specialty"].value_counts().nlargest(10).index
df["medical_specialty"] = df["medical_specialty"].where(
    df["medical_specialty"].isin(top_specialties), "Other"
)

def map_diagnosis(code):
    if pd.isna(code):
        return "Missing"
    code = str(code)
    if code.startswith("V") or code.startswith("E"):
        return "Other"
    try:
        val = float(code)
    except ValueError:
        return "Other"
    if 390 <= val <= 459 or val == 785:
        return "Circulatory"
    elif 460 <= val <= 519 or val == 786:
        return "Respiratory"
    elif 520 <= val <= 579 or val == 787:
        return "Digestive"
    elif 250 <= val < 251:
        return "Diabetes"
    elif 800 <= val <= 999:
        return "Injury"
    elif 710 <= val <= 739:
        return "Musculoskeletal"
    elif 580 <= val <= 629 or val == 788:
        return "Genitourinary"
    elif 140 <= val <= 239:
        return "Neoplasms"
    else:
        return "Other"

for c in ["diag_1", "diag_2", "diag_3"]:
    df[c] = df[c].apply(map_diagnosis)

df["race"] = df["race"].fillna("Missing")

constant_cols = [c for c in df.columns if df[c].nunique() == 1]
print("Dropping constant columns:", constant_cols)
df.drop(columns=constant_cols, inplace=True)

# readmitted มี 3 ค่า: 'NO', '<30', '>30' -> ตั้งเป็น binary classification
#   1 = readmitted ภายใน 30 วัน ('<30')
#   0 = ไม่ readmit หรือ readmit หลัง 30 วัน ('NO', '>30')
df["readmitted"] = df["readmitted"].apply(lambda x: 1 if x == "<30" else 0)
print("\nTarget distribution:\n", df["readmitted"].value_counts(normalize=True))

remaining_na = df.isna().sum()
print("\nRemaining missing values (should be empty):\n", remaining_na[remaining_na > 0])
print("\nFinal shape:", df.shape)

assert "visit_period" in df.columns, "visit_period หายไป เช็คว่ารันหัวข้อ 4 แล้วหรือยัง"
assert set(df["readmitted"].unique()) <= {0, 1}, "readmitted ยังไม่ถูก binarize"

df.to_csv("data/diabetic_data_hazard.csv", index=False)
print("\nSaved: data/diabetic_data_hazard.csv")


Dropping constant columns: ['examide', 'citoglipton']

Target distribution:
 readmitted
0    0.886108
1    0.113892
Name: proportion, dtype: float64

Remaining missing values (should be empty):
 Series([], dtype: int64)

Final shape: (99340, 49)

Saved: data/diabetic_data_hazard.csv


## 6. Encode + Train/Test Split (GroupShuffleSplit ตาม patient_nbr)

ใช้ spec เดียวกับ `03_model_preparation.ipynb` เป๊ะ (`test_size=0.2,
random_state=42`) เพื่อให้ผลลัพธ์เทียบกับ Stacking Ensemble เดิมได้ตรงๆ

⚠️ **จุดที่แก้จากเวอร์ชันก่อนหน้า**: `visit_period` เดิมเป็น string
(`"1","2","3","4+"`) ถ้าปล่อยให้ถูกจัดเป็น categorical แล้ว LabelEncode
ตามปกติ มันจะ (1) ไม่รับประกันว่าลำดับตัวเลขตรงกับความหมายจริง และ (2) ไม่ถูก
StandardScaler แตะเลย ทำให้ scale ไม่ตรงกับฟีเจอร์ตัวเลขอื่น จนโมเดล
LogisticRegression ไม่ converge — จึง**map เป็นตัวเลข ordinal เอง** แล้วย้าย
เข้า `numeric_cols` ให้ถูก scale ด้วย เหมือนที่ทำกับ `age`

In [ ]:
df = pd.read_csv("data/diabetic_data_hazard.csv")
print("Loaded shape:", df.shape)
assert "visit_period" in df.columns, "visit_period หายไป — เช็คว่าโหลดไฟล์ถูกไฟล์รึเปล่า"

TARGET_COL = "readmitted"
GROUP_COL = "patient_nbr"

X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL]

feature_cols = [c for c in X.columns if c != GROUP_COL]
categorical_cols = X[feature_cols].select_dtypes(include=["object"]).columns.tolist()
numeric_cols = [c for c in feature_cols if c not in categorical_cols]

# age เป็น ordinal -> map เป็นตัวเลขตำแหน่งกลางช่วง แล้วย้ายเข้า numeric_cols
age_map = {
    "[0-10)": 5, "[10-20)": 15, "[20-30)": 25, "[30-40)": 35,
    "[40-50)": 45, "[50-60)": 55, "[60-70)": 65, "[70-80)": 75,
    "[80-90)": 85, "[90-100)": 95,
}
X["age"] = X["age"].map(age_map)
categorical_cols.remove("age")
numeric_cols.append("age")

# visit_period เป็น ordinal เช่นกัน -> map เป็นตัวเลขแล้วย้ายเข้า numeric_cols
visit_period_map = {"1": 1, "2": 2, "3": 3, "4+": 4}
X["visit_period"] = X["visit_period"].map(visit_period_map)
categorical_cols.remove("visit_period")
numeric_cols.append("visit_period")

# ที่เหลือใช้ Label Encoding (พอสำหรับ logistic regression + เร็วกว่า one-hot)
label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))
    label_encoders[col] = le

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=df[GROUP_COL]))

X_train, X_test = X.iloc[train_idx].copy(), X.iloc[test_idx].copy()
y_train, y_test = y.iloc[train_idx].copy(), y.iloc[test_idx].copy()

train_patients = set(df.loc[train_idx, GROUP_COL])
test_patients = set(df.loc[test_idx, GROUP_COL])
print(f"Patient overlap train/test: {len(train_patients & test_patients)} (ต้องเป็น 0)")
print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")

X_train.drop(columns=[GROUP_COL], inplace=True)
X_test.drop(columns=[GROUP_COL], inplace=True)

scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()
X_train_scaled[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test_scaled[numeric_cols] = scaler.transform(X_test[numeric_cols])

print("visit_period อยู่ใน numeric_cols แล้ว:", "visit_period" in numeric_cols)
print("y_train unique values (ต้องมีแค่ 0,1):", sorted(y_train.unique()))


C:\Users\ACER\AppData\Local\Temp\ipykernel_8012\2252879697.py:1: DtypeWarning: Columns (19) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("data/diabetic_data_hazard.csv")


Loaded shape: (99340, 49)
Patient overlap train/test: 0 (ต้องเป็น 0)
Train shape: (79567, 48), Test shape: (19773, 48)
visit_period อยู่ใน numeric_cols แล้ว: True
y_train unique values (ต้องมีแค่ 0,1): [np.int64(0), np.int64(1)]


## 7. Model A (มี visit_period) vs Model B (ไม่มี, ablation)

ใช้ `class_weight="balanced"` เหมือนที่สรุปไว้แล้วใน `05_improved_pipeline4`
ว่าดีกว่า SMOTE variants สำหรับ dataset นี้ (ไม่ resample ข้อมูล แค่ปรับ
น้ำหนัก class ตอน fit)

Model B ตัด `visit_period`/`visit_number` ออก เพื่อดูว่าการใส่ duration
dependence เข้าไปช่วยอะไรจริงหรือเปล่า

`max_iter=2000` เพียงพอแล้วหลังจาก scale `visit_period` ถูกต้อง — ถ้ายังไม่
converge หลังแก้ scale แล้ว ค่อยพิจารณาเพิ่ม max_iter หรือเปลี่ยน solver

In [ ]:
hazard_model = LogisticRegression(class_weight="balanced", max_iter=2000, random_state=42)
hazard_model.fit(X_train_scaled, y_train)
proba_hazard = hazard_model.predict_proba(X_test_scaled)[:, 1]

no_time_cols = [c for c in X_train_scaled.columns if c not in ["visit_period", "visit_number"]]
notime_model = LogisticRegression(class_weight="balanced", max_iter=2000, random_state=42)
notime_model.fit(X_train_scaled[no_time_cols], y_train)
proba_notime = notime_model.predict_proba(X_test_scaled[no_time_cols])[:, 1]
print("Fitted both models.")


Fitted both models.


## 8. Evaluation — ROC-AUC / PR-AUC / Brier / Cost-Optimal Threshold (5:1)

ใช้ cost-based threshold search แบบเดียวกับที่ใช้กับ Stacking Ensemble ใน
`05_improved_pipeline4.ipynb` (cost_fn:cost_fp = 5:1) เพื่อให้เทียบกันได้
apples-to-apples

In [ ]:
def evaluate(y_true, proba, cost_fn=5, cost_fp=1, label=""):
    roc = roc_auc_score(y_true, proba)
    pr = average_precision_score(y_true, proba)
    brier = brier_score_loss(y_true, proba)

    thresholds = np.linspace(0.01, 0.99, 197)
    costs = []
    for t in thresholds:
        pred = (proba >= t).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_true, pred).ravel()
        costs.append(cost_fn * fn + cost_fp * fp)
    best_t = thresholds[np.argmin(costs)]
    pred = (proba >= best_t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred).ravel()
    recall = tp / (tp + fn)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0

    print(f"\n--- {label} ---")
    print(f"ROC-AUC        : {roc:.4f}")
    print(f"PR-AUC         : {pr:.4f}")
    print(f"Brier score    : {brier:.4f}")
    print(f"Cost-optimal threshold (5:1): {best_t:.3f}")
    print(f"  Recall       : {recall:.4f}")
    print(f"  Precision    : {precision:.4f}")
    print(f"  Confusion matrix: TN={tn} FP={fp} FN={fn} TP={tp}")
    return dict(roc_auc=roc, pr_auc=pr, brier=brier, threshold=best_t,
                recall=recall, precision=precision, tn=tn, fp=fp, fn=fn, tp=tp)

res_hazard = evaluate(y_test, proba_hazard, label="Discrete-Time Hazard Model (with visit_period)")
res_notime = evaluate(y_test, proba_notime, label="Ablation: same model WITHOUT visit_period")



--- Discrete-Time Hazard Model (with visit_period) ---
ROC-AUC        : 0.6570
PR-AUC         : 0.2092
Brier score    : 0.2287
Cost-optimal threshold (5:1): 0.570
  Recall       : 0.3668
  Precision    : 0.2096
  Confusion matrix: TN=14425 FP=3104 FN=1421 TP=823

--- Ablation: same model WITHOUT visit_period ---
ROC-AUC        : 0.6539
PR-AUC         : 0.2102
Brier score    : 0.2291
Cost-optimal threshold (5:1): 0.600
  Recall       : 0.2906
  Precision    : 0.2193
  Confusion matrix: TN=15208 FP=2321 FN=1592 TP=652


## 9. Baseline Hazard Shape — Observed vs. Predicted by Visit Period

นี่คือ output ที่เป็นหัวใจของ discrete-time hazard model: กราฟ/ตาราง hazard
ตามลำดับ period แสดงว่าความเสี่ยงเปลี่ยนไปอย่างไรเมื่อผู้ป่วยสะสมจำนวนครั้ง
ที่มา รพ. มากขึ้น (duration dependence)

⚠️ predicted hazard สูงกว่า observed rate มากเพราะใช้
`class_weight="balanced"` ซึ่งดันความน่าจะเป็นขึ้นเพื่อแก้ imbalance —
**ยังไม่ได้ calibrate** ถ้าจะรายงานตัวเลขความเสี่ยงจริงในเปเปอร์ ควรห่อด้วย
`CalibratedClassifierCV` (isotonic) ก่อน — ที่นี่ปล่อยไว้แบบ raw เพื่อให้เห็น
**อันดับความเสี่ยง (ranking)** ระหว่าง period ชัดๆ ก่อน

In [ ]:
test_df = df.loc[test_idx].copy()
test_df["predicted_hazard"] = proba_hazard
test_df["actual"] = y_test.values

hazard_by_period = test_df.groupby("visit_period").agg(
    n=("actual", "size"),
    observed_rate=("actual", "mean"),
    predicted_hazard=("predicted_hazard", "mean"),
).reindex(["1", "2", "3", "4+"])

print(hazard_by_period)


                  n  observed_rate  predicted_hazard
visit_period                                        
1             13998       0.088227          0.423167
2              3273       0.142071          0.537212
3              1181       0.191363          0.608677
4+             1321       0.240727          0.697846


## 10. Coefficients (Interpretability)

In [ ]:
coef_df = pd.DataFrame({
    "feature": X_train_scaled.columns,
    "coefficient": hazard_model.coef_[0],
}).sort_values("coefficient", key=abs, ascending=False)
print(coef_df.head(15).to_string(index=False))


                 feature  coefficient
          chlorpropamide    -0.655272
              tolazamide    -0.655076
             tolbutamide    -0.298032
     glipizide-metformin    -0.283143
             diabetesMed     0.243620
                acarbose    -0.237518
        number_inpatient     0.193638
            troglitazone    -0.190426
discharge_disposition_id     0.161724
               metformin    -0.147016
             glimepiride    -0.142729
 metformin-rosiglitazone    -0.126089
            visit_period     0.102481
                miglitol     0.095574
        number_diagnoses     0.095132


⚠️ **ข้อควรระวังในการตีความ**: ยาบางตัวที่มีผู้ใช้น้อยมาก (sparse
category) ผ่าน LabelEncoder ที่ mapping เป็นตัวเลขโดยพลการ (ไม่ใช่ ordinal จริง)
อาจมีค่าสัมประสิทธิ์สูงผิดปกติ — **ไม่ควรตีความว่าเป็นตัวทำนายที่แข็งแรงจริง**
ตัวที่น่าเชื่อถือกว่าคือฟีเจอร์ที่มีขนาดตัวอย่างมากพอ เช่น `number_inpatient`,
`discharge_disposition_id`, `visit_period`

## เทียบกับ Stacking Ensemble เดิม (จาก `05_improved_pipeline4.ipynb`)

ตัวเลข Stacking Ensemble ด้านล่างเป็นค่าที่รันไว้แล้วใน `05_improved_pipeline4.ipynb`
(ที่ threshold cost-optimal 5:1 = 0.610) — ใส่เป็นค่าคงที่ไว้เพื่อเทียบ ไม่ได้รันโมเดล
Stacking ซ้ำในไฟล์นี้ ถ้าไปรัน `05_improved_pipeline4.ipynb` ใหม่แล้วตัวเลขเปลี่ยน
ให้กลับมาแก้ dict `stacking_ensemble_ref` ในเซลล์ถัดไปให้ตรงกันด้วย

In [ ]:
# ค่าอ้างอิงจาก 05_improved_pipeline4.ipynb (Stacking Ensemble, cost-optimal threshold=0.610)
stacking_ensemble_ref = {
    "threshold": 0.610,
    "roc_auc": 0.6763,
    "pr_auc": 0.2268,
    "recall": 0.3436,
    "precision": 0.2378,
}

comparison_df = pd.DataFrame({
    "Hazard (with visit_period)": {
        "ROC-AUC": res_hazard["roc_auc"],
        "PR-AUC": res_hazard["pr_auc"],
        "Recall @ cost-optimal": res_hazard["recall"],
        "Precision @ cost-optimal": res_hazard["precision"],
        "Threshold": res_hazard["threshold"],
    },
    "Hazard (no visit_period)": {
        "ROC-AUC": res_notime["roc_auc"],
        "PR-AUC": res_notime["pr_auc"],
        "Recall @ cost-optimal": res_notime["recall"],
        "Precision @ cost-optimal": res_notime["precision"],
        "Threshold": res_notime["threshold"],
    },
    "Stacking Ensemble": {
        "ROC-AUC": stacking_ensemble_ref["roc_auc"],
        "PR-AUC": stacking_ensemble_ref["pr_auc"],
        "Recall @ cost-optimal": stacking_ensemble_ref["recall"],
        "Precision @ cost-optimal": stacking_ensemble_ref["precision"],
        "Threshold": stacking_ensemble_ref["threshold"],
    },
}).round(4)

print(comparison_df.to_string())

                          Hazard (with visit_period)  Hazard (no visit_period)  Stacking Ensemble
ROC-AUC                                       0.6570                    0.6539             0.6763
PR-AUC                                        0.2092                    0.2102             0.2268
Recall @ cost-optimal                         0.3668                    0.2906             0.3436
Precision @ cost-optimal                      0.2096                    0.2193             0.2378
Threshold                                     0.5700                    0.6000             0.6100


## 11. Save Results

In [ ]:
os.makedirs("models", exist_ok=True)
joblib.dump(
    {
        "hazard_model": hazard_model,
        "notime_model": notime_model,
        "scaler": scaler,
        "label_encoders": label_encoders,
        "numeric_cols": numeric_cols,
        "categorical_cols": categorical_cols,
        "feature_names": X_train_scaled.columns.tolist(),
        "res_hazard": res_hazard,
        "res_notime": res_notime,
        "hazard_by_period": hazard_by_period,
        "coef_df": coef_df,
        "y_test": y_test,
        "proba_hazard": proba_hazard,
    },
    "models/hazard_results.pkl",
)
print("Saved: models/hazard_results.pkl")


Saved: models/hazard_results.pkl
